# Destination Projection — Historical Backtest & Error Diagnostics

Compares the Destination-Adjusted Projection model's **actual production output** (re-scored point-in-time per historical season) against **actual realized** per-game stats for real historical transfers, then mines residuals for systematic bias patterns.

Full design: `docs/models/destination_projection_backtest_plan.md`. This notebook does the *interactive* half of that plan (§7b clustering, cohort review, plots) — the non-interactive residual computation/logging lives in `scripts/run_destination_backtest.py`, matching every other model's interactive-notebook-vs-`run_*.py`-script split in this repo.

**Not duplicated here:** `fit_role_usage_model` CV (see `scheme_fit_scorer.ipynb`'s precedent / `destination_projection.py`'s own `compute_cohort_validation`), the residual load/compute functions (imported from `portalpoint.modeling.destination_backtest`, not reimplemented).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from portalpoint.modeling import destination_backtest as db
from portalpoint.modeling.destination_projection import MODEL_VERSION
from portalpoint.modeling.io import get_sync_engine

pd.set_option("display.max_columns", 50)
engine = get_sync_engine()

## 1. Load population, actual outcomes, projected outcomes, residuals

Reuses `destination_backtest.py`'s pure functions directly — no logic redefined here.

In [ ]:
# min_dest_season=2023, not 2022 — 2022 confirmed infeasible (real
# run_playing_time.py failure): barttorvik data starts at season 2021, so
# target_season=2022's Playing Time model can't get the >=2 prior seasons it
# hard-requires. See destination_projection_backtest_plan.md §13.
MIN_SEASON, MAX_SEASON = 2023, 2026

population_df = db.load_backtest_population(engine, MIN_SEASON, MAX_SEASON)
print(f"Backtest population: {len(population_df):,} matched historical transfers")
population_df["dest_season"].value_counts().sort_index()

In [ ]:
# Readiness check — which seasons actually have destination-mode player_projections
# rows already? (see scripts/run_destination_backtest.py --backfill for filling gaps)
from sqlalchemy import text

seasons = sorted(population_df["dest_season"].unique().tolist())
with engine.connect() as conn:
    present = {
        int(r[0])
        for r in conn.execute(
            text(
                "SELECT DISTINCT season FROM player_projections "
                "WHERE projection_mode = 'destination' AND model_version = :mv "
                "AND season = ANY(:seasons)"
            ),
            {"mv": MODEL_VERSION, "seasons": seasons},
        ).fetchall()
    }
missing = sorted(s for s in seasons if s not in present)
print(f"Seasons ready: {sorted(present)}")
print(f"Seasons missing (need backfill first): {missing}")

In [ ]:
actual_df = db.load_actual_outcomes(engine, population_df)
projected_df = db.load_projected_outcomes(engine, population_df, MODEL_VERSION)
residual_df = db.compute_residuals(actual_df, projected_df)

print(f"Actual outcomes (games-played floor applied): {len(actual_df):,}")
print(f"Projected outcomes found: {len(projected_df):,}")
print(f"Backtest rows with both (real n for this analysis): {len(residual_df):,}")
residual_df.head()

## 2. §7a — Cohort splits (known categories, extends `compute_cohort_validation`'s slice definitions)

Position, archetype (M1, joined at *source* season — what kind of player this was heading into the portal), and tier direction (same derivation as the production pipeline's own `assign_competition_tiers`). Cheap, interpretable — do this first.

In [ ]:
enriched = db.enrich_with_cohorts(engine, population_df)
residual_df = residual_df.merge(
    enriched[["player_id", "dest_school_id", "dest_season", "archetype_label", "tier_direction", "position"]],
    on=["player_id", "dest_school_id", "dest_season"],
    how="left",
)

overall = db.summarize_residuals(residual_df)
print("=== Overall ===")
overall

In [ ]:
by_position = db.summarize_residuals(residual_df, group_by="position")
pd.DataFrame(by_position).T

In [ ]:
by_archetype = db.summarize_residuals(residual_df, group_by="archetype_label")
pd.DataFrame(by_archetype).T

In [ ]:
by_tier_direction = db.summarize_residuals(residual_df, group_by="tier_direction")
pd.DataFrame(by_tier_direction).T

## 3. §7b — Unsupervised clustering on residual vectors (exploratory)

Per-player residual vector across all 6 stats, scaled, k-means. Purpose: surface a bias mode nobody thought to slice by ahead of time — not a metric to optimize. Cluster count/interpretation need human review each run (same posture as M1/M2 clustering notebooks) — **inspect the silhouette/inertia plot and label the clusters yourself before trusting any conclusion.**

In [ ]:
residual_cols = [f"residual_{stat}" for stat in db.BACKTEST_STATS]
cluster_input = residual_df[residual_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_input.values)

# K-sweep — inspect inertia before picking a final K (same discipline as player_clustering.ipynb)
inertias = []
K_RANGE = range(2, 9)
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

pd.Series(inertias, index=list(K_RANGE), name="inertia")

In [ ]:
# Pick K from the elbow above, then inspect each cluster's mean residual vector —
# this is the "what bias pattern does this cluster represent" step, human-read.
K_FINAL = 4  # placeholder — set from the inertia plot above

km_final = KMeans(n_clusters=K_FINAL, n_init=10, random_state=42)
cluster_labels = km_final.fit_predict(X_scaled)

clustered = cluster_input.copy()
clustered["cluster"] = cluster_labels
cluster_profile = clustered.groupby("cluster")[residual_cols].mean()
cluster_profile["n"] = clustered.groupby("cluster").size()
cluster_profile

## 4. Does the §7a cohort split already explain what §7b clustering finds?

Cross-tab the discovered clusters against position/archetype/tier_direction — if a cluster maps cleanly onto an existing cohort, clustering didn't find anything new. If a cluster cuts *across* existing cohorts, that's the useful finding — a bias mode nobody had a name for yet.

In [ ]:
cohort_check = residual_df.loc[cluster_input.index, ["position", "archetype_label", "tier_direction"]].copy()
cohort_check["cluster"] = cluster_labels

print("Cluster x Position:")
display(pd.crosstab(cohort_check["cluster"], cohort_check["position"]))

print("\nCluster x Tier direction:")
display(pd.crosstab(cohort_check["cluster"], cohort_check["tier_direction"]))

print("\nCluster x Archetype (top 3 per cluster):")
for c in sorted(cohort_check["cluster"].unique()):
    top = cohort_check.loc[cohort_check["cluster"] == c, "archetype_label"].value_counts().head(3)
    print(f"  cluster {c}: {top.to_dict()}")